In [3]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

os.chdir('/Users/sidhantnarang/student-survival-predictor-au/student-survival-predictor-au')
os.makedirs('models', exist_ok=True)

df = pd.read_csv('data/synthetic/students.csv')
print(df.shape)
df.head()

(2000, 12)


,city,rent_weekly,income_weekly,transport_weekly,grocery_weekly,tuition_weekly,study_hours,work_hours,commute_hours,total_expenses,savings_weekly,financial_stress
0,Brisbane,437,232,32,137,199,47,15,1.672915,805,-573,1
1,Melbourne,555,326,42,76,270,32,9,2.577818,943,-617,1
2,Brisbane,372,347,39,69,264,44,16,0.538706,744,-397,1
3,Brisbane,591,570,56,74,165,31,0,1.182293,886,-316,1
4,Melbourne,420,550,47,115,244,38,17,0.506136,826,-276,1


In [4]:
#Train the model
features = ['rent_weekly', 'income_weekly', 'transport_weekly',
            'grocery_weekly', 'tuition_weekly', 'study_hours',
            'work_hours', 'commute_hours']

X = df[features]
y = df['financial_stress']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.83      0.71      0.77         7
           1       0.99      1.00      1.00       393

    accuracy                           0.99       400
   macro avg       0.91      0.86      0.88       400
weighted avg       0.99      0.99      0.99       400



In [5]:
#Save the model
joblib.dump(model, 'models/stress_predictor.pkl')
print("✅ Model saved to models/stress_predictor.pkl")

# Verify it loads back correctly
loaded = joblib.load('models/stress_predictor.pkl')
print("✅ Model loads correctly, type:", type(loaded))

✅ Model saved to models/stress_predictor.pkl
✅ Model loads correctly, type: <class 'xgboost.sklearn.XGBClassifier'>


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
}

results = []
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds)*100, 1),
        'F1 Score': round(f1_score(y_test, preds), 3),
        'ROC AUC':  round(roc_auc_score(y_test, preds), 3),
    })

import pandas as pd
results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
print(results_df.to_string(index=False))
# Save best model (XGBoost usually wins)
import joblib
best = models['XGBoost']
joblib.dump(best, 'models/stress_predictor.pkl')

              Model  Accuracy  F1 Score  ROC AUC
Logistic Regression      99.5     0.997    0.927
            XGBoost      99.2     0.996    0.856
      Random Forest      98.8     0.994    0.643


['models/stress_predictor.pkl']